In [1]:
# ============================================================
# GIÁO TRÌNH: THỊ GIÁC MÁY: TỪ XỬ LÝ ẢNH ĐẾN HỌC SÂU
# BÀI CODE MINH HỌA: ViT - Head-Level Grad-CAM trên Attention Matrix
# Chương/Mục liên quan: Chương 7/8 - Vision Transformer và giải thích mô hình học sâu
# ============================================================

# ============================================================
# MÔ TẢ
# ============================================================

# Mục đích:
# - Minh họa cách phân tích attention head trong Vision Transformer.
# - Tính gradient của lớp dự đoán theo attention tensor.
# - Tạo Head-Level Grad-CAM cho từng head theo công thức:
#   HeadCAM_h = ReLU(A_h * G_h)
# - Chọn 4 head tiêu biểu và hiển thị attention/Grad-CAM tương ứng.

# Sau khi chạy code, người học cần:
# 1. Quan sát được lưới token 14x14 của ViT-B/16.
# 2. Hiểu cách lấy attention từ token CLS đến các patch ảnh.
# 3. Hiểu sự khác nhau giữa attention thô và Head-Level Grad-CAM.

# Input:
# - Ảnh màu từ GitHub:
#   https://github.com/lthavnu/cv-book/raw/main/images/samoyed.jpg
# - Mô hình pretrained: google/vit-base-patch16-224

# Output:
# - Ảnh đầu vào
# - Lưới token
# - Mean attention
# - Mean Head-Level Grad-CAM
# - 4 head attention tiêu biểu
# - 4 head Grad-CAM tiêu biểu
# - Bảng điểm các head được chọn

# Lưu ý
# Đoạn code này được xây dựng với sự hỗ trợ của công cụ AI.
# Giảng viên đã đọc, kiểm tra và hiệu chỉnh nhằm bảo đảm tính chính xác,
# tính sư phạm và sự phù hợp với nội dung lý thuyết trong giáo trình.

# ============================================================
# 1. CÀI ĐẶT VÀ IMPORT THƯ VIỆN
# ============================================================

!pip install -q transformers torch torchvision pillow matplotlib requests

import math
import requests
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from io import BytesIO

import torch
import torch.nn.functional as F

from transformers import AutoImageProcessor, ViTForImageClassification

# ============================================================
# 2. KHAI BÁO THAM SỐ
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "google/vit-base-patch16-224"
IMAGE_SOURCE = "https://github.com/lthavnu/cv-book/raw/main/images/samoyed.jpg"

# ViT-Base có 12 block, đánh số từ 0 đến 11
BLOCK_INDEX = 11

# Số head tiêu biểu cần hiển thị
TOPK_HEADS = 4

# Độ trong suốt khi chồng heatmap lên ảnh gốc
ALPHA = 0.45

print("DEVICE =", DEVICE)

# ============================================================
# 3. TẢI DỮ LIỆU ĐẦU VÀO VÀ LOAD MODEL
# ============================================================

def load_image(image_source: str) -> Image.Image:
    if image_source.startswith("http://") or image_source.startswith("https://"):
        r = requests.get(image_source, timeout=30)
        r.raise_for_status()
        img = Image.open(BytesIO(r.content)).convert("RGB")
    else:
        img = Image.open(image_source).convert("RGB")
    return img


def load_model_and_processor(model_name: str):
    processor = AutoImageProcessor.from_pretrained(model_name)

    try:
        model = ViTForImageClassification.from_pretrained(
            model_name,
            output_attentions=True
        ).to(DEVICE).eval()
    except Exception:
        model = ViTForImageClassification.from_pretrained(
            model_name,
            output_attentions=True,
            attn_implementation="eager"
        ).to(DEVICE).eval()

    return processor, model


def forward_with_attentions(model, processor, image: Image.Image):
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    outputs = model(**inputs, output_attentions=True)

    if outputs.attentions is None:
        raise RuntimeError("Không lấy được attentions. Hãy thử attn_implementation='eager'.")

    return inputs, outputs

# ============================================================
# 4. CÁC HÀM TIỆN ÍCH CHO HEATMAP
# ============================================================

def normalize_map(m):
    m = m.astype(np.float32)
    m = m - m.min()

    if m.max() > 1e-8:
        m = m / m.max()

    return m


def resize_mask_nearest(mask_2d: np.ndarray, out_hw):
    t = torch.tensor(mask_2d, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=out_hw, mode="nearest")
    return t[0, 0].cpu().numpy()


def overlay_heatmap(image_np: np.ndarray, mask: np.ndarray, alpha=0.45):
    mask = normalize_map(mask)
    cmap = plt.get_cmap("jet")
    heat = cmap(mask)[..., :3]

    out = (1 - alpha) * image_np + alpha * heat
    return np.clip(out, 0, 1)


def draw_patch_grid(ax, image_shape_hw, grid_size, color="white", linewidth=1.0, alpha=0.8):
    h, w = image_shape_hw
    step_y = h / grid_size
    step_x = w / grid_size

    for i in range(1, grid_size):
        ax.axhline(i * step_y, color=color, linewidth=linewidth, alpha=alpha)
        ax.axvline(i * step_x, color=color, linewidth=linewidth, alpha=alpha)

# ============================================================
# 5. CHUYỂN ATTENTION TỪ TOKEN CLS SANG PATCH MASK
# ============================================================

def cls_to_patch_mask_from_matrix(mat_2d: torch.Tensor):
    # mat_2d có kích thước (tokens, tokens)
    # Lấy hàng CLS và bỏ cột CLS để thu được attention từ CLS đến các patch
    cls_to_patches = mat_2d[0, 1:]

    num_patches = cls_to_patches.shape[0]
    grid_size = int(math.sqrt(num_patches))

    assert grid_size * grid_size == num_patches, "Số patch không tạo thành lưới vuông."

    mask = cls_to_patches.reshape(grid_size, grid_size).detach().cpu().numpy()
    return normalize_map(mask)


def mean_attention_to_patch_mask(attn_layer: torch.Tensor):
    # attn_layer có kích thước (1, heads, tokens, tokens)
    # Gộp attention trung bình theo các head
    attn_mean = attn_layer[0].mean(dim=0)
    return cls_to_patch_mask_from_matrix(attn_mean)


def head_attention_to_patch_mask(attn_layer: torch.Tensor, head_idx: int):
    # Lấy attention thô của một head cụ thể
    attn_head = attn_layer[0, head_idx]
    return cls_to_patch_mask_from_matrix(attn_head)

# ============================================================
# 6. TÍNH HEAD-LEVEL GRAD-CAM TRÊN ATTENTION
# ============================================================

def head_gradcam_to_patch_mask(attn_layer: torch.Tensor, grad_layer: torch.Tensor, head_idx: int):
    # A là attention matrix của một head
    A = attn_layer[0, head_idx]

    # G là gradient của lớp dự đoán theo attention matrix tương ứng
    G = grad_layer[0, head_idx]

    # Dòng này tương ứng với công thức HeadCAM_h = ReLU(A_h * G_h)
    cam_mat = torch.relu(A * G)

    # Lấy hàng CLS -> patches để chuyển thành bản đồ 2D
    return cls_to_patch_mask_from_matrix(cam_mat)


def gradcam_strength(mask):
    # Điểm cường độ Head-CAM, dùng để tham khảo khi chọn head tiêu biểu
    x = mask.reshape(-1).astype(np.float32)
    return float(x.mean() + x.max())

# ============================================================
# 7. CHẤM ĐIỂM ĐỂ CHỌN HEAD TIÊU BIỂU
# ============================================================

def concentration_score(mask):
    x = mask.reshape(-1).astype(np.float32)
    x = x / (x.sum() + 1e-8)

    k = max(1, int(0.10 * len(x)))
    topk = np.partition(x, -k)[-k:]

    return float(topk.sum())


def contrast_score(mask):
    x = mask.reshape(-1).astype(np.float32)
    return float(np.percentile(x, 95) - np.percentile(x, 50))


def center_bias_score(mask):
    h, w = mask.shape
    yy, xx = np.mgrid[0:h, 0:w]

    cy, cx = (h - 1) / 2.0, (w - 1) / 2.0

    dist = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    dist = dist / (dist.max() + 1e-8)

    weight = 1.0 - dist

    return float((mask * weight).sum() / (mask.sum() + 1e-8))


def representative_score(attn_mask, gradcam_mask):
    # Điểm chọn head dựa trên độ tập trung, độ tương phản, độ gần trung tâm và cường độ Grad-CAM
    s1 = concentration_score(attn_mask)
    s2 = contrast_score(attn_mask)
    s3 = center_bias_score(attn_mask)
    s4 = gradcam_strength(gradcam_mask)

    return 0.30 * s1 + 0.20 * s2 + 0.10 * s3 + 0.40 * s4


def cosine_similarity(a, b):
    a = a.reshape(-1).astype(np.float32)
    b = b.reshape(-1).astype(np.float32)

    na = np.linalg.norm(a) + 1e-8
    nb = np.linalg.norm(b) + 1e-8

    return float(np.dot(a, b) / (na * nb))


def select_topk_diverse_heads(attn_layer, grad_layer, topk=4, diversity_threshold=0.92):
    num_heads = attn_layer.shape[1]
    info = []

    for h in range(num_heads):
        attn_mask = head_attention_to_patch_mask(attn_layer, h)
        cam_mask = head_gradcam_to_patch_mask(attn_layer, grad_layer, h)
        score = representative_score(attn_mask, cam_mask)

        info.append({
            "head_index": h,
            "attn_mask": attn_mask,
            "cam_mask": cam_mask,
            "score": float(score),
            "cam_strength": float(gradcam_strength(cam_mask)),
        })

    order = sorted(info, key=lambda x: x["score"], reverse=True)

    selected = []

    for cand in order:
        ok = True

        for s in selected:
            sim_attn = cosine_similarity(cand["attn_mask"], s["attn_mask"])
            sim_cam = cosine_similarity(cand["cam_mask"], s["cam_mask"])

            if max(sim_attn, sim_cam) > diversity_threshold:
                ok = False
                break

        if ok:
            selected.append(cand)

        if len(selected) == topk:
            break

    if len(selected) < topk:
        for cand in order:
            if cand not in selected:
                selected.append(cand)

            if len(selected) == topk:
                break

    return selected, info

# ============================================================
# 8. XỬ LÝ CHÍNH: FORWARD VÀ BACKPROP
# ============================================================

def compute_attention_gradients(model, processor, image, block_index):
    # Xóa gradient cũ
    model.zero_grad(set_to_none=True)

    # Chạy forward để lấy logits và attention
    inputs, outputs = forward_with_attentions(model, processor, image)

    # Tính xác suất dự đoán bằng softmax
    probs = torch.softmax(outputs.logits, dim=-1)

    pred_idx = int(probs.argmax(dim=-1)[0].item())
    pred_prob = float(probs[0, pred_idx].item())
    pred_label = model.config.id2label[pred_idx]

    # Lấy attention tensor tại block cần phân tích
    attn_layer = outputs.attentions[block_index]

    # Giữ gradient trên attention tensor
    attn_layer.retain_grad()

    # Lấy score của lớp được dự đoán
    score = outputs.logits[0, pred_idx]

    # Dòng này tương ứng với bước backprop để tính gradient d(score)/d(attention)
    score.backward()

    if attn_layer.grad is None:
        raise RuntimeError("Không lấy được gradient của attention tensor.")

    grad_layer = attn_layer.grad

    return {
        "inputs": inputs,
        "outputs": outputs,
        "attn_layer": attn_layer,
        "grad_layer": grad_layer,
        "pred_idx": pred_idx,
        "pred_prob": pred_prob,
        "pred_label": pred_label,
    }

# ============================================================
# 9. HIỂN THỊ KẾT QUẢ
# ============================================================

def visualize_head_attention_and_gradcam(
    image_source=IMAGE_SOURCE,
    model_name=MODEL_NAME,
    block_index=BLOCK_INDEX,
    topk_heads=TOPK_HEADS,
    alpha=ALPHA
):
    image = load_image(image_source)
    image_np = np.asarray(image).astype(np.float32) / 255.0

    processor, model = load_model_and_processor(model_name)

    pack = compute_attention_gradients(model, processor, image, block_index)

    attn_layer = pack["attn_layer"]
    grad_layer = pack["grad_layer"]

    pred_label = pack["pred_label"]
    pred_prob = pack["pred_prob"]

    mean_attn_mask_small = mean_attention_to_patch_mask(attn_layer)
    grid_size = mean_attn_mask_small.shape[0]

    mean_attn_mask_img = resize_mask_nearest(mean_attn_mask_small, image_np.shape[:2])
    mean_attn_overlay = overlay_heatmap(image_np, mean_attn_mask_img, alpha=alpha)

    selected, all_info = select_topk_diverse_heads(
        attn_layer,
        grad_layer,
        topk=topk_heads,
        diversity_threshold=0.92
    )

    mean_cam_small = np.mean([x["cam_mask"] for x in all_info], axis=0)
    mean_cam_small = normalize_map(mean_cam_small)

    mean_cam_img = resize_mask_nearest(mean_cam_small, image_np.shape[:2])
    mean_cam_overlay = overlay_heatmap(image_np, mean_cam_img, alpha=alpha)

    fig, axes = plt.subplots(3, 4, figsize=(20, 14))
    axes = np.array(axes).reshape(3, 4)

    axes[0, 0].imshow(image_np)
    axes[0, 0].set_title("Ảnh đầu vào")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(image_np)
    draw_patch_grid(axes[0, 1], image_np.shape[:2], grid_size)
    axes[0, 1].set_title(f"Lưới token ({grid_size}×{grid_size})")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(mean_attn_overlay)
    draw_patch_grid(axes[0, 2], image_np.shape[:2], grid_size, linewidth=0.5, alpha=0.55)
    axes[0, 2].set_title(f"Block {block_index + 1} - mean attention")
    axes[0, 2].axis("off")

    axes[0, 3].imshow(mean_cam_overlay)
    draw_patch_grid(axes[0, 3], image_np.shape[:2], grid_size, linewidth=0.5, alpha=0.55)
    axes[0, 3].set_title(f"Block {block_index + 1} - mean head Grad-CAM")
    axes[0, 3].axis("off")

    for i in range(4):
        ax = axes[1, i]

        if i < len(selected):
            head_idx = selected[i]["head_index"]

            attn_mask_small = selected[i]["attn_mask"]
            attn_mask_img = resize_mask_nearest(attn_mask_small, image_np.shape[:2])
            attn_overlay = overlay_heatmap(image_np, attn_mask_img, alpha=alpha)

            ax.imshow(attn_overlay)
            draw_patch_grid(ax, image_np.shape[:2], grid_size, linewidth=0.5, alpha=0.55)
            ax.set_title(
                f"Head {head_idx + 1} - attention\nscore={selected[i]['score']:.3f}",
                fontsize=11
            )
            ax.axis("off")
        else:
            ax.axis("off")

    for i in range(4):
        ax = axes[2, i]

        if i < len(selected):
            head_idx = selected[i]["head_index"]

            cam_mask_small = selected[i]["cam_mask"]
            cam_mask_img = resize_mask_nearest(cam_mask_small, image_np.shape[:2])
            cam_overlay = overlay_heatmap(image_np, cam_mask_img, alpha=alpha)

            ax.imshow(cam_overlay)
            draw_patch_grid(ax, image_np.shape[:2], grid_size, linewidth=0.5, alpha=0.55)
            ax.set_title(
                f"Head {head_idx + 1} - head Grad-CAM\ncam={selected[i]['cam_strength']:.3f}",
                fontsize=11
            )
            ax.axis("off")
        else:
            ax.axis("off")

    plt.suptitle(
        f"ViT | Block {block_index + 1} | Dự đoán: {pred_label} ({pred_prob:.4f})",
        fontsize=16,
        y=0.995
    )

    plt.tight_layout()
    plt.show()

    print(f"\nTop {len(selected)} head tiêu biểu ở block {block_index + 1}:")

    for x in selected:
        print(
            f"  Head {x['head_index'] + 1:2d} | "
            f"representative_score = {x['score']:.4f} | "
            f"gradcam_strength = {x['cam_strength']:.4f}"
        )

    return {
        "image": image,
        "pack": pack,
        "selected": selected,
        "all_info": all_info,
    }

# ============================================================
# 10. KIỂM TRA KẾT QUẢ
# ============================================================

result = visualize_head_attention_and_gradcam()

assert result["pack"]["attn_layer"] is not None
assert result["pack"]["grad_layer"] is not None
assert len(result["selected"]) == TOPK_HEADS

print("\nKiểm tra hoàn tất:")
print("- Đã lấy được attention tensor.")
print("- Đã lấy được gradient theo attention tensor.")
print("- Đã chọn đủ số head tiêu biểu.")

Output hidden; open in https://colab.research.google.com to view.